In [13]:
from datetime import datetime, timezone
import pandas as pd
from simple_dyson_pool import SimpleDysonPool, Note_forward, Note_reverse

def format_number(number, precision=4):
    # Round to the specified precision
    rounded = round(number, precision)
    # Format with the specified precision and remove trailing zeros
    result = f"{rounded:.{precision}f}".rstrip("0").rstrip(".")
    return result


def display_pool_state(pool, label):
    snapshot = pool.snapshot(day=0, price=2000)  # Simplified day to a constant
    df = pd.DataFrame([{
        "ETH Reserve": format_number(snapshot["reserve_eth"]),
        "USDC Reserve": format_number(snapshot["reserve_usdc"]),
        "K": format_number(snapshot["k"]),
        "w": pool.w,
    }])
    display(df)


def test_forward_deposit_and_withdraw():
    result = []
    input_values = [
        (1, 0),
        (0, 2000),
        (1, 2000),
        (1.1, 2000),
    ]

    for in0, in1 in input_values:
        pool = SimpleDysonPool(init_x=100.0, init_y=200000.0)
        (nid, note0, note1, Q, x, y, q) = pool.deposit(in0, in1)
        note = pool.notes_forward[nid]

        assert note.in0 == in0, f"Expected in0={in0}, got {note.in0}"
        assert note.in1 == in1, f"Expected in1={in1}, got {note.in1}"

        amt0, amt1 = pool.withdraw(nid)

        result.append(
            {
                "Input0": format_number(note.in0),
                "Input1": format_number(note.in1),
                "Note0": format_number(note.note0),
                "Note1": format_number(note.note1),
                "Q": format_number(Q),
                "x_after_deposit": format_number(x),
                "y_after_deposit": format_number(y),
                "q_after_deposit": format_number(q),
                "Withdrawn0": format_number(amt0),
                "Withdrawn1": format_number(amt1),
            }
        )
    df = pd.DataFrame(result)
    display(df)

def test_reverse_deposit_and_exercise():
    results = []
    # Test cases with different m, n and exercise options
    testcase_values = [
        (1, 0, "put"),
        (0, 2000, "call"),
        (1, 2000, "put"),
        (1.1, 2000, "put")
    ]

    for m, n, exercise_option in testcase_values:
        pool = SimpleDysonPool(init_x=100.0, init_y=200000.0)
        pool.deposit(100, 200000)  # Initial deposit to make q > 0
        (nid, strike, delta_x, delta_y, Q, x, y, q) = pool.reverse_deposit(m, n)

        note = pool.notes_reverse[nid]

        out0, out1 = pool.exercise_option(nid, exercise_option)
        # Assert
        assert note.m == m, f"Expected m={m}, got {note.m}"
        assert note.n == n, f"Expected n={n}, got {note.n}"
        if exercise_option == "call":
            assert out1 == note.n 
            assert out0 == note.n / note.strike
        elif exercise_option == "put":
            assert out0 == note.m
            assert out1 == note.m * note.strike
        else:
            raise ValueError(f"Unknown option type: {exercise_option}")

        results.append({
            "m(Token0 User Received)": format_number(note.m),
            "n(Token1 User Received)": format_number(note.n),
            "Delta X": format_number(delta_x),
            "Delta Y": format_number(delta_y),
            "Strike": format_number(strike),
            "Q": format_number(Q),
            "x_after_deposit": format_number(x),
            "y_after_deposit": format_number(y),
            "q_after_deposit": format_number(q),
            "Exercise Call (swap1 in, swap0 out)": f"({format_number(note.n)}, {format_number(note.n / note.strike)})",
            "Exercise Put (swap0 in, swap1 out)": f"({format_number(note.m)}, {format_number(note.m * note.strike)})",
        })

    df = pd.DataFrame(results)
    display(df)

# 正向雙幣

In [14]:
test_forward_deposit_and_withdraw()

,Input0,Input1,Note0,Note1,Q,x_after_deposit,y_after_deposit,q_after_deposit,Withdrawn0,Withdrawn1
0,1,0,1,1990.0621,44.6101,101,200000,44.6101,0.7503,496.8967
1,0,2000,0.995,2000,44.6101,100,202000,44.6101,0.2484,1500.6219
2,1,2000,2,4000,89.4427,101,202000,89.4427,1,2000
3,1.1,2000,2.1,4199.901,93.9137,101.1,202000,93.9137,1.0988,2002.3326


# 反向雙幣

In [15]:
test_reverse_deposit_and_exercise()

,m(Token0 User Received),n(Token1 User Received),Delta X,Delta Y,Strike,Q,x_after_deposit,y_after_deposit,q_after_deposit,"Exercise Call (swap1 in, swap0 out)","Exercise Put (swap0 in, swap1 out)"
0,1,0,-1,2010.0503,2010.0503,-44.7774,199,402010.0503,8899.4945,"(0, 0)","(1, 2010.0503)"
1,0,2000,1.005,-2000,1990,-44.7774,201.005,398000,8899.4945,"(2000, 1.005)","(0, 0)"
2,1,2000,0,0,2000,-89.4427,200,400000,8854.8292,"(2000, 1)","(1, 2000)"
3,1.1,2000,-0.1005,201.1061,2001.0055,-93.9154,199.8995,400201.1061,8850.3565,"(2000, 0.9995)","(1.1, 2201.1061)"
